# Suspended rotary pendulum — angular-acceleration input

균일한 bar pendulum, 저속 근사, 입력 $u=\ddot\phi$를 사용하여 suspended 상태의 전달함수와 1초 각가속도 pulse 응답을 계산합니다.

## 1. 모델의 출발점: torque와 angular acceleration
직선 운동의 $F=ma$에 대응하여 회전 운동은
$$\tau=J\alpha=J\ddot q$$
입니다. 균일한 bar pendulum의 질량을 $m$, 전체 길이를 $l$이라 하고 한쪽 끝을 pivot으로 두면
$$l_c=\frac l2,\qquad J_p=\frac13ml^2.$$
Rotary arm 각도는 $\phi$, motor 축에서 pendulum pivot까지의 거리는 $r$입니다.

완전한 비선형식에는 $\dot\phi^2$, $\dot\theta^2$, $\dot\phi\dot\theta$ 항이 있지만 여기서는 **각속도가 작다**고 가정하여 이 2차 속도항들을 무시합니다. 그러면 downward 기준 전역 pendulum 각도 $\vartheta$에 대해
$$\frac13ml^2\ddot\vartheta+\frac12mrl\cos\vartheta\,\ddot\phi+\frac12mgl\sin\vartheta=0.$$
$J_p=\frac13ml^2$로 나누면
$$\ddot\vartheta+\frac{3r}{2l}\cos\vartheta\,\ddot\phi+\frac{3g}{2l}\sin\vartheta=0.$$
여기서 질량 $m$은 소거됩니다. 즉 $u=\ddot\phi$를 이상적인 입력으로 주는 모델에서 pendulum 각도 응답은 $m$에 직접 의존하지 않습니다. 다만 실제 step motor가 그 각가속도를 만들기 위해 필요한 torque에는 $m$이 영향을 줍니다.

편의를 위해
$$a=\frac{3g}{2l},\qquad b=\frac{3r}{2l},\qquad u=\ddot\phi$$
로 둡니다.

## 2. Suspended 평형점 선형화
아래로 매달린 위치를 local angle $\theta=0$으로 둡니다. 작은 각도에서
$$\sin\theta\approx\theta,\qquad \cos\theta\approx1$$
이므로
$$\boxed{\ddot\theta+a\theta=-bu}.$$
초기조건을 0으로 두고 Laplace transform하면
$$\boxed{\frac{\Theta(s)}{U(s)}=-\frac{b}{s^2+a}}.$$
$l=0.235\,\mathrm m$, $r=0.14\,\mathrm m$이면
$$a=62.6170,\qquad b=0.893617,$$
따라서
$$\boxed{\frac{\Theta(s)}{U(s)}=-\frac{0.893617}{s^2+62.6170}}.$$
마찰을 생략했으므로 pole은 $s=\pm j\sqrt a$이고 이상 모델은 감쇠 없이 진동합니다.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Plot text is intentionally English/ASCII so it renders correctly on any OS/Jupyter setup.
# This also fixes the Unicode minus issue if Korean labels are added later.
plt.rcParams["axes.unicode_minus"] = False

g=9.81
l=0.235
r=0.14
a=3*g/(2*l)
b=3*r/(2*l)
print(f"a={a:.6f} 1/s^2, b={b:.6f}")


## 3. 1초 각가속도 pulse
$$u(t)=\begin{cases}1\ \mathrm{rad/s^2},&0\le t<1\\0,&t\ge1\end{cases}$$
를 사용합니다. 주의할 점은 1초 뒤 $u=0$으로 만드는 것은 **arm을 정지시키는 명령이 아니라 가속을 멈추는 것**입니다. 따라서 이상 모델에서는 $\dot\phi(1)\approx1\,\mathrm{rad/s}$가 된 뒤 그 속도를 계속 유지합니다.

In [ ]:
u0=1.0; pulse_time=1.0; t_end=5.0
def command(t): return u0 if t < pulse_time else 0.0
def rhs(t,x):
    theta,theta_dot,phi,phi_dot=x
    u=command(t)
    return [theta_dot,-a*theta-b*u,phi_dot,u]
t_eval=np.linspace(0,t_end,5001)
sol=solve_ivp(rhs,(0,t_end),[0,0,0,0],t_eval=t_eval,max_step=.002,rtol=1e-9,atol=1e-11)
t=sol.t; theta,theta_dot,phi,phi_dot=sol.y
u=np.where(t<pulse_time,u0,0.0)
print(f"max |theta| = {np.rad2deg(np.max(np.abs(theta))):.4f} deg")
print(f"final phi_dot = {phi_dot[-1]:.4f} rad/s")

In [ ]:
fig,ax=plt.subplots(4,1,figsize=(9,9),sharex=True)
ax[0].plot(t,u); ax[0].set_ylabel("arm accel [rad/s^2]")
ax[1].plot(t,np.rad2deg(theta)); ax[1].set_ylabel("theta [deg]")
ax[2].plot(t,phi_dot); ax[2].set_ylabel("phi_dot [rad/s]")
ax[3].plot(t,np.rad2deg(phi)); ax[3].set_ylabel("phi [deg]"); ax[3].set_xlabel("time [s]")
for axy in ax: axy.grid(True)
fig.suptitle("Suspended pendulum: 1 s arm-acceleration pulse")
fig.tight_layout()
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/suspended_acceleration_pulse_response.png",dpi=160,bbox_inches="tight")
plt.show()

## 4. 해석
- 0–1 s 동안 arm acceleration이 pivot을 가속하여 pendulum을 흔듭니다.
- 1 s 이후에는 외부 arm acceleration input이 사라져도, 마찰이 없는 suspended pendulum은 자체 진동을 계속합니다.
- 실제 stepper에서는 $u$를 적분해 $\dot\phi_{cmd}$를 만들고 pulse frequency로 변환하며, 속도/가속도 제한과 감속 구간을 추가해야 합니다.